In [14]:
!pip install pandas mlxtend

/opt/miniconda3/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=38693) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


In [15]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv('Downloads/retail_data - retail_data.csv')

df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['Description'] = df['Description'].astype(str).str.strip()

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,40513.35139,2.55,17850.0,Spain
1,536365,71053,WHITE METAL LANTERN,6,40513.35139,3.39,17850.0,Spain
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,40513.35139,2.75,17850.0,Spain
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,40513.35139,3.39,17850.0,Spain
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,40513.35139,3.39,17850.0,Spain


In [16]:
cancelled_orders = df[df['InvoiceNo'].str.contains('C', na=False)]
print("--- Sample of Cancelled Orders ---")
display(cancelled_orders.head())

df_clean = df[~df['InvoiceNo'].str.contains('C', na=False)].copy()



--- Sample of Cancelled Orders ---


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,40513.40347,27.50,14527.0,Ireland
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,40513.40903,4.65,15311.0,United Kingdom
235,C536391,10/2/1961,PLASTERS IN TIN CIRCUS PARADE,-12,40513.43333,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,40513.43333,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,40513.43333,0.29,17548.0,United Kingdom


In [17]:
df_clean = df_clean[df_clean['Description'].str.upper() != 'POSTAGE']

df_clean = df_clean.dropna(subset=['CustomerID'])

df_clean = df_clean[df_clean['Quantity'] > 0]

print(f"\nDataset shape after cleaning: {df_clean.shape}")


Dataset shape after cleaning: (621, 8)


In [18]:
total_unique_countries = df_clean['Country'].nunique()
print(f"Total number of unique countries: {total_unique_countries}")

df_uk = df_clean[df_clean['Country'] == 'United Kingdom'].copy()
print(f"UK dataset shape: {df_uk.shape}")

Total number of unique countries: 11
UK dataset shape: (512, 8)


In [19]:
def prepare_basket(dataframe):
    basket = (dataframe.groupby(['InvoiceNo', 'Description'])['Quantity']
              .sum().unstack().reset_index().fillna(0)
              .set_index('InvoiceNo'))
    def encode_units(x):
        if x <= 0: return 0
        if x >= 1: return 1
            
    return basket.applymap(encode_units)



In [23]:
def prepare_basket(dataframe):
    
    basket = (dataframe.groupby(['InvoiceNo', 'Description'])['Quantity']
              .sum().unstack().reset_index().fillna(0)
              .set_index('InvoiceNo'))
    
    
    def encode_units(x):
        if x <= 0:
            return 0
        if x >= 1:
            return 1
            
    
    basket_sets = basket.map(encode_units)
    return basket_sets

In [24]:
cancelled_orders = df[df['InvoiceNo'].astype(str).str.contains('C', na=False)]
print("Sample of Cancelled Orders")
display(cancelled_orders.head())

df_clean = df[~df['InvoiceNo'].astype(str).str.contains('C', na=False)].copy()

Sample of Cancelled Orders


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,40513.40347,27.50,14527.0,Ireland
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,40513.40903,4.65,15311.0,United Kingdom
235,C536391,10/2/1961,PLASTERS IN TIN CIRCUS PARADE,-12,40513.43333,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,40513.43333,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,40513.43333,0.29,17548.0,United Kingdom


In [25]:
df_clean = df_clean[df_clean['Description'].astype(str).str.upper() != 'POSTAGE']

df_clean = df_clean.dropna(subset=['CustomerID'])

df_clean = df_clean[df_clean['Quantity'] > 0]

print(f"\nDataset shape after cleaning: {df_clean.shape}")


Dataset shape after cleaning: (621, 8)


In [26]:
def prepare_basket(dataframe):
    basket = (dataframe.groupby(['InvoiceNo', 'Description'])['Quantity']
              .sum().unstack().reset_index().fillna(0)
              .set_index('InvoiceNo'))
    
    
    basket.columns.name = None
    
    basket_sets = basket > 0
            
    return basket_sets

In [27]:
basket_full = prepare_basket(df_clean)
basket_uk = prepare_basket(df_uk)

print("Full Basket Shape:", basket_full.shape)
print("UK Basket Shape:", basket_uk.shape)

Full Basket Shape: (50, 394)
UK Basket Shape: (36, 342)
